# MwC 02 — subject-level 26-ROI extraction

Extract and export the 12-cortical + 14-limbic subject-level MwC values.

- Formal analysis universe: 12 cortical + 14 limbic ROIs (26 nodes)
- Cross-network connectivity: 12 × 14 = 168 edges
- Pipeline: `mwc`; step 2 of 5
- Run mode: standalone; all prerequisite steps load automatically in the same kernel.


In [1]:
from pathlib import Path
import os

_repo = Path(os.environ.get("METAFL_ROOT", Path.cwd())).resolve()
if not (_repo / "pipeline_steps").is_dir() and (_repo.parent / "pipeline_steps").is_dir():
    _repo = _repo.parent
if not (_repo / "pipeline_steps").is_dir():
    raise FileNotFoundError("Set METAFL_ROOT to the repository root")
os.environ["METAFL_ROOT"] = str(_repo)
os.chdir(_repo)

def _run_step(relative_path):
    _path = _repo / "pipeline_steps" / relative_path
    _code = compile(_path.read_text(encoding="utf-8"), str(_path), "exec")
    exec(_code, get_ipython().user_ns)
_run_step("mwc_01_setup_recompute.py")


,input,size_kib
0,03_compute_normative_sc_mwc.py,12.4
1,session_gcmi_pet_AAL3_144.npz,19663.6
2,normative_SC_top10_mask.npz,53.7
3,subject_level_corticolimbic26_normativeSC10_Mw...,139.7
4,subject_level_AAL3_144_normativeSC10_MwC_wide....,333.6
5,validation_report.json,0.2
6,corticolimbic_12x14_ROI_definition.csv,0.6
7,clinical_data.xlsx,12.6
8,subject_level_corticolimbic26_age_sex_adjusted...,154.9


{
  "full_144": "./outputs/mwc/recomputed_normative_sc/subject_level_AAL3_144_normativeSC10_MwC_wide.csv.gz",
  "selected_26": "./outputs/mwc/recomputed_normative_sc/subject_level_corticolimbic26_normativeSC10_MwC_wide.csv",
  "validation": "./outputs/mwc/recomputed_normative_sc/calculation_validation_report.json"
}
PASS: AAL3-144 normative-SC-constrained MwC recomputed before 26-ROI extraction


,check,passed
0,sessions_291,True
1,retained_AAL3_nodes_144,True
2,fixed_dTOR_normative_SC_edges_1030,True
3,gcmi_floor_bits_0p01,True
4,row_normalization_error_below_1e_12,True
5,all_144_nodes_computed_before_26_extraction,True
6,isolated_nodes_remain_missing,True
7,full_144_matches_validated_reference,True
8,selected_26_matches_validated_reference,True


## Preamble


In [2]:
"""Auto-extracted pipeline step: MwC 02 — subject-level 26-ROI extraction.

Do not edit generated copies independently of the source notebooks.
"""


'Auto-extracted pipeline step: MwC 02 — subject-level 26-ROI extraction.\n\nDo not edit generated copies independently of the source notebooks.\n'

## Source 6


In [3]:
roi = pd.read_csv(ROI_FILE).reset_index(drop=True)
roi["node_index"] = np.arange(1, len(roi) + 1)
roi["ROI_full"] = [f"AAL3_{int(row.AAL3_ID):03d}_{row.ROI}" for row in roi.itertuples()]
assert len(roi) == N_ROIS

with AAL_VALIDATION_FILE.open(encoding="utf-8") as stream:
    aal_validation = json.load(stream)
assert aal_validation["status"] == "PASS"
assert aal_validation["checks"]["retained_nodes_144"]
assert aal_validation["checks"]["fixed_structural_edges_1030"]
assert mwc_calculation_validation["checks"]["all_144_nodes_computed_before_26_extraction"]

mwc_source = pd.read_csv(MWC_WIDE_FILE)
full_source = pd.read_csv(MWC_FULL_FILE)
assert len(mwc_source) == len(full_source) == 291
key_columns = ["subject_key", "session"]
assert mwc_source[key_columns].equals(full_source[key_columns])

mwc_columns = [f"MwC_{name}" for name in roi.ROI]
full_columns = [
    f"MwC_AAL3_{int(aal3_id):03d}_{name}"
    for aal3_id, name in zip(roi.AAL3_ID, roi.ROI)
]
assert set(mwc_columns).issubset(mwc_source)
assert set(full_columns).issubset(full_source)
all144_extraction_error = float(np.nanmax(np.abs(
    mwc_source[mwc_columns].to_numpy(float)
    - full_source[full_columns].to_numpy(float)
)))
assert all144_extraction_error < 1e-10


def _metadata_norm_key(value):
    if pd.isna(value):
        return None
    text = str(value).strip()
    match = re.fullmatch(r'P(\d{3})', text)
    if not match:
        raise ValueError(f'Expected a deidentified P### subject key, got {text!r}')
    return f'P{int(match.group(1)):03d}'


clinical_all_mwc = pd.read_excel(CLINICAL_FILE)
clinical_all_mwc['subject_key'] = clinical_all_mwc['subject_key'].map(_metadata_norm_key)
clinical_all_mwc['age'] = pd.to_numeric(clinical_all_mwc['age'], errors='coerce')
clinical_all_mwc['sex'] = pd.to_numeric(clinical_all_mwc['sex'], errors='coerce')
clinical_all_mwc['education'] = pd.to_numeric(clinical_all_mwc['education_years'], errors='coerce')
clinical_all_mwc['hdrs_pre'] = pd.to_numeric(clinical_all_mwc['hdrs17_pre'], errors='coerce')
clinical_all_mwc['hdrs_post'] = pd.to_numeric(clinical_all_mwc['hdrs17_post'], errors='coerce')
clinical_lookup = clinical_all_mwc.set_index('subject_key')
expected_clinical_counts = {"HC": 23, "nTRD": 25, "TRD1": 35, "TRD2": 104}
assert clinical_all_mwc["analysis_cohort"].value_counts().to_dict() == expected_clinical_counts

mwc_source = mwc_source.copy()
mwc_source['subject_key'] = mwc_source['subject_key'].map(_metadata_norm_key)
mwc_source.insert(0, "session_row", np.arange(len(mwc_source), dtype=int))
clinical_session_mask = mwc_source["subject_key"].isin(clinical_lookup.index)
mwc_source = mwc_source.loc[clinical_session_mask].reset_index(drop=True)

for column in ['analysis_cohort', 'analysis_arm', 'age', 'sex', 'education', 'hdrs_pre', 'hdrs_post']:
    mapped = mwc_source['subject_key'].map(clinical_lookup[column])
    mwc_source[column] = mapped

metadata_columns = [
    "session_row", "subject_key", "session", "analysis_cohort", "analysis_arm",
    "age", "sex", "education", "hdrs_pre", "hdrs_post",
]
mwc_wide = mwc_source[metadata_columns + mwc_columns].copy()
observed_node_mwc = mwc_wide[mwc_columns].to_numpy(float)

matrix_frame = pd.DataFrame(
    observed_node_mwc, index=mwc_wide.session_row, columns=roi.node_index,
)
matrix_frame.index.name = "session_row"
mwc_long = (
    matrix_frame.reset_index()
    .melt(id_vars="session_row", var_name="node_index", value_name="MwC")
    .merge(
        roi[["node_index", "network", "AAL3_ID", "ROI", "ROI_full"]],
        on="node_index", how="left", validate="many_to_one",
    )
    .merge(mwc_wide[metadata_columns], on="session_row", how="left", validate="many_to_one")
    .sort_values(["session_row", "node_index"])
    .reset_index(drop=True)
)

ALIASES = {
    "Frontal_Sup_Medial": "dmPFC",
    "Frontal_Med_Orb": "vmPFC",
    "OFCmed": "medOFC",
}


def display_roi_name(roi_name: str) -> str:
    for source_name, alias in ALIASES.items():
        if roi_name == source_name or roi_name.startswith(source_name + "_"):
            return roi_name.replace(source_name, alias, 1)
    return roi_name


mwc_long["display_roi"] = mwc_long.ROI.map(display_roi_name)
mwc_wide.to_csv(OUTPUT_DIR / "subject_level_corticolimbic26_normativeSC10_MwC_wide.csv", index=False)
mwc_long.to_csv(OUTPUT_DIR / "subject_level_corticolimbic26_normativeSC10_MwC_long.csv", index=False)

# HC-referenced deviations are regenerated from the normative-SC-constrained MwC values.
baseline_hc = mwc_wide.loc[
    mwc_wide.session.eq("ses-01") & mwc_wide.analysis_cohort.eq("HC")
].copy()
assert len(baseline_hc) == 23
hc_mean = baseline_hc[mwc_columns].mean()
hc_sd = baseline_hc[mwc_columns].std(ddof=1)
assert np.isfinite(hc_sd).all() and (hc_sd > 0).all()
baseline_groups = mwc_wide.loc[
    mwc_wide.session.eq("ses-01")
    & mwc_wide.analysis_cohort.isin(["nTRD", "TRD1"])
].copy()
assert baseline_groups.groupby("analysis_cohort").size().to_dict() == {"TRD1": 35, "nTRD": 25}
deviation = baseline_groups[[
    "session_row", "subject_key", "session", "analysis_cohort", "analysis_arm",
    "age", "sex", "education",
]].copy()
for source_column, roi_name in zip(mwc_columns, roi.ROI):
    deviation[f"MwC_deviation_{roi_name}"] = (
        baseline_groups[source_column].to_numpy(float) - hc_mean[source_column]
    ) / hc_sd[source_column]
deviation["display_group"] = deviation.analysis_cohort.map({"nTRD": "nTRD", "TRD1": "TRD"})
deviation.to_csv(
    OUTPUT_DIR / "subject_level_nTRD_TRD1_corticolimbic26_normativeSC10_MwC_HC_reference_deviation.csv",
    index=False,
)

display(pd.DataFrame({
    "quantity": [
        "upstream paired sessions", "clinical sessions", "ROIs per session",
        "long-table rows", "all-144 to 26 extraction maximum absolute error",
    ],
    "value": [291, len(mwc_wide), N_ROIS, len(mwc_long), all144_extraction_error],
}))
display(roi[["node_index", "network", "AAL3_ID", "ROI"]])


,quantity,value
0,upstream paired sessions,291.0
1,clinical sessions,291.0
2,ROIs per session,26.0
3,long-table rows,7566.0
4,all-144 to 26 extraction maximum absolute error,0.0


,node_index,network,AAL3_ID,ROI
0,1,cortical,3,Frontal_Sup_2_L
1,2,cortical,4,Frontal_Sup_2_R
2,3,cortical,5,Frontal_Mid_2_L
3,4,cortical,6,Frontal_Mid_2_R
4,5,cortical,19,Frontal_Sup_Medial_L
5,6,cortical,20,Frontal_Sup_Medial_R
6,7,cortical,21,Frontal_Med_Orb_L
7,8,cortical,22,Frontal_Med_Orb_R
8,9,cortical,25,OFCmed_L
9,10,cortical,26,OFCmed_R


## Results preview

Key numerical tables and figures generated by this step are displayed below. The full-resolution files remain under `outputs/`.


In [4]:
from io import BytesIO
import json as _preview_json
import pandas as _preview_pd
from PIL import Image as PILImage
from IPython.display import Image as NotebookImage, Markdown, display

_inline_table_specs = [{'title': 'Subject-level 26-ROI MwC values', 'path': 'outputs/mwc/subject_level_corticolimbic26_normativeSC10_MwC_wide.csv', 'max_rows': 10}]
_inline_figure_paths = []

display(Markdown("### Key tables"))
for _table_spec in _inline_table_specs:
    _relative_table = _table_spec["path"]
    _table_path = _repo / _relative_table
    if not _table_path.is_file():
        raise FileNotFoundError(_table_path)
    if _table_path.suffix.lower() == ".csv":
        _preview_table = _preview_pd.read_csv(_table_path)
    elif _table_path.suffix.lower() in {".xlsx", ".xls"}:
        _preview_table = _preview_pd.read_excel(
            _table_path, sheet_name=_table_spec.get("sheet", 0)
        )
    elif _table_path.suffix.lower() == ".json":
        _payload = _preview_json.loads(_table_path.read_text(encoding="utf-8"))
        _checks = _payload.get("checks", _payload)
        _preview_table = _preview_pd.DataFrame(
            {"item": list(_checks), "value": list(_checks.values())}
        ) if isinstance(_checks, dict) else _preview_pd.json_normalize(_payload)
    else:
        raise ValueError(f"Unsupported table format: {_table_path.suffix}")
    if _table_spec.get("query"):
        _preview_table = _preview_table.query(_table_spec["query"]).reset_index(drop=True)
    _max_rows = int(_table_spec.get("max_rows", 40))
    display(Markdown(
        f"**{_table_spec['title']}**  \n"
        f"`{_relative_table}` — {len(_preview_table)} rows × {len(_preview_table.columns)} columns"
    ))
    with _preview_pd.option_context("display.max_columns", 40, "display.max_colwidth", 60):
        display(_preview_table.head(_max_rows))

if _inline_figure_paths:
    display(Markdown("### Figures"))
for _relative_figure in _inline_figure_paths:
    _figure_path = _repo / _relative_figure
    if not _figure_path.is_file():
        raise FileNotFoundError(_figure_path)
    with PILImage.open(_figure_path) as _source_image:
        _preview_image = _source_image.convert("RGB")
        _preview_image.thumbnail((1600, 1200), PILImage.Resampling.LANCZOS)
        _preview_buffer = BytesIO()
        _preview_image.save(_preview_buffer, format="PNG", optimize=True)
    display(Markdown(f"**`{_relative_figure}`**"))
    display(NotebookImage(data=_preview_buffer.getvalue(), format="png"))


### Key tables

**Subject-level 26-ROI MwC values**  
`outputs/mwc/subject_level_corticolimbic26_normativeSC10_MwC_wide.csv` — 291 rows × 36 columns

,session_row,subject_key,session,analysis_cohort,analysis_arm,age,sex,education,hdrs_pre,hdrs_post,MwC_Frontal_Sup_2_L,MwC_Frontal_Sup_2_R,MwC_Frontal_Mid_2_L,MwC_Frontal_Mid_2_R,MwC_Frontal_Sup_Medial_L,MwC_Frontal_Sup_Medial_R,MwC_Frontal_Med_Orb_L,MwC_Frontal_Med_Orb_R,MwC_OFCmed_L,MwC_OFCmed_R,MwC_OFClat_L,MwC_OFClat_R,MwC_Hippocampus_L,MwC_Hippocampus_R,MwC_ParaHippocampal_L,MwC_ParaHippocampal_R,MwC_Amygdala_L,MwC_Amygdala_R,MwC_Caudate_L,MwC_Caudate_R,MwC_Putamen_L,MwC_Putamen_R,MwC_Pallidum_L,MwC_Pallidum_R,MwC_ACC_sub_L,MwC_ACC_sub_R
0,0,P001,ses-01,HC,NaN,59,1,12,2,NaN,103.922892,105.529612,101.879852,101.487458,108.921148,107.959736,103.009156,102.345495,94.645280,86.784057,102.688569,93.846355,82.208319,85.195706,80.586743,80.576804,73.597522,79.871090,102.526244,98.443795,103.582837,102.971389,105.939612,110.065581,105.971408,107.020040
1,1,P002,ses-01,HC,NaN,62,1,12,6,NaN,101.278283,103.204782,103.654194,100.426101,103.671349,101.561013,100.793775,100.871650,93.705047,94.880345,100.231136,87.977064,90.161782,88.942062,94.519824,91.651530,75.164448,77.239513,103.619257,98.494269,97.580001,103.860703,99.210593,102.857991,99.722688,103.550875
2,2,P003,ses-01,HC,NaN,69,1,13,5,NaN,99.568804,99.744117,101.425926,101.333307,100.553242,99.851871,95.449307,96.389450,92.604345,91.345010,101.076258,93.700075,84.035785,80.984500,87.350605,80.693916,78.718209,77.576940,105.746578,98.451941,94.974574,104.861740,120.951004,115.230436,96.884409,97.663885
3,3,P004,ses-01,HC,NaN,68,1,6,0,NaN,106.517449,103.245353,105.659464,99.488587,108.453584,105.691208,102.880663,102.080771,92.238900,86.445247,96.315010,85.586351,84.080077,87.990266,78.948461,81.502133,78.312514,99.726856,111.365216,98.413903,103.343166,103.162745,118.944000,108.924106,108.392767,NaN
4,4,P005,ses-01,HC,NaN,78,1,6,0,NaN,98.547962,98.827946,102.233667,100.235075,97.925720,99.124974,95.604778,97.771566,97.809602,103.860316,97.863217,92.883533,73.947946,81.843433,80.598687,93.290469,76.054149,75.143682,104.488344,98.399190,100.066746,104.077585,110.879763,116.080610,105.300030,102.836220
5,5,P006,ses-01,HC,NaN,58,0,12,7,NaN,101.622227,102.599862,103.889984,104.337230,100.563799,98.689905,97.693899,93.513951,104.835020,100.831721,102.645016,94.250172,82.124507,82.503803,86.995732,90.129229,81.174714,86.948992,104.279083,100.971056,99.328212,105.807105,101.624479,106.392949,110.935305,99.684538
6,6,P007,ses-01,HC,NaN,53,1,14,5,NaN,101.435367,101.398334,105.129788,101.399550,104.668590,101.297334,105.093491,108.569560,110.758969,103.177187,102.734254,97.774750,97.035411,93.697480,98.326568,87.546172,75.692844,92.693398,99.182517,102.180792,97.036955,103.915479,110.777617,111.731966,109.493708,86.880394
7,7,P008,ses-01,HC,NaN,46,1,14,0,NaN,102.469336,103.541987,107.766442,104.971399,103.856544,104.416085,101.249365,101.476820,108.071991,103.256022,102.472669,95.678502,92.766648,89.250129,90.461154,88.302629,85.866235,88.097399,108.048208,99.776968,104.188947,107.854889,125.035280,109.905486,112.900255,108.376366
8,8,P009,ses-01,HC,NaN,59,1,12,0,NaN,100.176256,101.708547,104.728259,103.513254,109.596997,104.735482,100.192589,100.897939,98.419635,96.912744,99.711223,95.667863,86.960948,81.371593,81.739184,83.535806,91.530454,86.406257,102.480773,101.875553,105.903510,104.696628,118.592370,113.885600,104.383348,101.140929
9,9,P010,ses-01,HC,NaN,38,1,16,1,NaN,105.599436,105.472854,106.502668,104.788933,104.142165,103.521580,97.507901,100.637991,107.977340,103.779960,110.025211,NaN,93.688890,90.391848,94.015525,88.866239,87.942428,82.797250,107.854316,102.174668,105.034529,102.536149,98.596266,110.925769,115.402642,109.157370
